# Notebook 03 — Model Building

**Project:** Fraud Detection & Strategy Analytics  
**Objective:** Train and compare three classification models for fraud detection, evaluate them on business-relevant metrics, and use SHAP to explain model predictions.

---

## Models
| # | Model | Role |
|---|-------|------|
| 1 | Logistic Regression | Interpretable baseline |
| 2 | Random Forest | Non-linear ensemble |
| 3 | XGBoost | State-of-the-art gradient boosting |

## Metrics
- **AUC-ROC** — overall discriminatory power
- **KS Statistic** — maximum separation between fraud / legit score distributions
- **Precision / Recall / F1** — at optimised threshold
- **SHAP** — global and local feature importance

---

## Outline
1. Load and split data
2. Build feature matrix
3. Train models
4. Evaluate and compare
5. ROC curves
6. Score distribution plots
7. SHAP feature importance
8. Select best model

In [ ]:
import sys
sys.path.insert(0, '..')

import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import shap

from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_curve, auc

from data.generate_data import generate_dataset
from src.data_processing import clean_data, split_data
from src.features import build_feature_matrix
from src.models import (
    get_logistic_regression,
    get_random_forest,
    get_xgboost,
    train_model,
    evaluate_model,
    compare_models,
    compute_shap_values,
    get_top_features,
)

sns.set_theme(style='whitegrid', font_scale=1.1)
plt.rcParams['figure.dpi'] = 110
print('Imports complete.')

## 1. Load and Split Data

In [ ]:
df_raw = generate_dataset(n=100_000)
df_clean = clean_data(df_raw)
train_df, val_df, test_df = split_data(df_clean)

print(f'Train: {len(train_df):,} | Val: {len(val_df):,} | Test: {len(test_df):,}')
print(f'Train fraud rate: {train_df["is_fraud"].mean():.2%}')
print(f'Test  fraud rate: {test_df["is_fraud"].mean():.2%}')

## 2. Build Feature Matrices

In [ ]:
DROP_COLS = ['transaction_id', 'timestamp', 'is_fraud', 'amount_bucket_label', 'label']

def prepare_X_y(df):
    df_eng = build_feature_matrix(df)
    feat_cols = [c for c in df_eng.columns if c not in DROP_COLS]
    X = df_eng[feat_cols].values
    y = df_eng['is_fraud'].values
    return X, y, feat_cols

X_train, y_train, feature_names = prepare_X_y(train_df)
X_val,   y_val,   _             = prepare_X_y(val_df)
X_test,  y_test,  _             = prepare_X_y(test_df)

# Scale for logistic regression
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_val_s   = scaler.transform(X_val)
X_test_s  = scaler.transform(X_test)

# Imbalance ratio for XGBoost
scale_pos_weight = (y_train == 0).sum() / (y_train == 1).sum()

print(f'Feature matrix shape: {X_train.shape}')
print(f'Feature count: {len(feature_names)}')
print(f'scale_pos_weight: {scale_pos_weight:.1f}')

## 3. Train Models

> ⏱ This cell may take 2–4 minutes to run (XGBoost and Random Forest are CPU-intensive).

In [ ]:
print('Training Logistic Regression…')
lr_model = train_model(get_logistic_regression(), X_train_s, y_train)
print('  Done.')

print('Training Random Forest…')
rf_model = train_model(get_random_forest(), X_train, y_train)
print('  Done.')

print('Training XGBoost…')
xgb_model = train_model(
    get_xgboost(scale_pos_weight=scale_pos_weight),
    X_train, y_train,
    X_val=X_val, y_val=y_val
)
print('  Done.')

## 4. Evaluate Models on Test Set

In [ ]:
# Get fraud probability predictions
lr_probs  = lr_model.predict_proba(X_test_s)[:, 1]
rf_probs  = rf_model.predict_proba(X_test)[:, 1]
xgb_probs = xgb_model.predict_proba(X_test)[:, 1]

# Evaluate each model
lr_metrics  = evaluate_model('Logistic Regression', y_test, lr_probs)
rf_metrics  = evaluate_model('Random Forest',       y_test, rf_probs)
xgb_metrics = evaluate_model('XGBoost',             y_test, xgb_probs)

comparison_df = compare_models([lr_metrics, rf_metrics, xgb_metrics])
print('=== Model Comparison (Test Set) ===')
print(comparison_df.to_string())

In [ ]:
# Visualise comparison
metrics_to_plot = ['auc_roc', 'ks_stat', 'precision', 'recall', 'f1']
plot_df = comparison_df[metrics_to_plot].reset_index()
plot_melted = plot_df.melt(id_vars='model', var_name='metric', value_name='score')

fig, ax = plt.subplots(figsize=(11, 5))
sns.barplot(data=plot_melted, x='metric', y='score', hue='model',
            palette='Set2', ax=ax)
ax.set_title('Model Performance Comparison — Test Set')
ax.set_ylabel('Score')
ax.set_xlabel('Metric')
ax.legend(title='Model')
ax.set_ylim(0, 1.05)
plt.tight_layout()
plt.show()

> **Interpretation:**  
> XGBoost consistently leads on AUC-ROC and KS statistic, confirming its superiority for tabular fraud data.  
> Logistic Regression performs respectably given its simplicity — useful as an explainable fallback or regulatory-required alternative.

## 5. ROC Curves

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))

for name, probs, color in [
    ('Logistic Regression', lr_probs,  'steelblue'),
    ('Random Forest',       rf_probs,  'seagreen'),
    ('XGBoost',             xgb_probs, 'tomato'),
]:
    fpr, tpr, _ = roc_curve(y_test, probs)
    roc_auc = auc(fpr, tpr)
    ax.plot(fpr, tpr, color=color, lw=2, label=f'{name} (AUC = {roc_auc:.4f})')

ax.plot([0, 1], [0, 1], 'k--', lw=1, label='Random classifier')
ax.set_xlabel('False Positive Rate')
ax.set_ylabel('True Positive Rate')
ax.set_title('ROC Curves — Test Set')
ax.legend(loc='lower right')
ax.set_xlim([0, 1])
ax.set_ylim([0, 1.02])
plt.tight_layout()
plt.show()

## 6. Score Distribution — Fraud vs. Legitimate

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

for ax, name, probs in zip(
    axes,
    ['Logistic Regression', 'Random Forest', 'XGBoost'],
    [lr_probs, rf_probs, xgb_probs],
):
    ax.hist(probs[y_test == 0], bins=60, alpha=0.6, color='steelblue',
            label='Legitimate', density=True)
    ax.hist(probs[y_test == 1], bins=60, alpha=0.7, color='tomato',
            label='Fraud', density=True)
    ax.set_title(name)
    ax.set_xlabel('Fraud Probability Score')
    ax.set_ylabel('Density')
    ax.legend(fontsize=9)

plt.suptitle('Score Distributions: Fraud vs Legitimate', fontsize=13)
plt.tight_layout()
plt.show()

> **Observation:** XGBoost produces the cleanest score separation — the fraud distribution is clearly shifted toward 1.0, and the legitimate distribution clusters near 0.  
> Good score separation is critical for strategy design in Notebook 04: clear gaps enable precise cutoff placement.

## 7. SHAP Feature Importance

SHAP (SHapley Additive exPlanations) provides a rigorous, game-theoretic decomposition of each prediction into per-feature contributions. This is critical for fraud strategy work because:
1. **Regulatory compliance** — we must be able to explain why a transaction was declined.
2. **Strategy design** — understanding which features drive scores guides rule creation.
3. **Model monitoring** — SHAP drift signals model degradation.

In [ ]:
print('Computing SHAP values for XGBoost (this may take ~30s)…')
shap_values, shap_explanation = compute_shap_values(
    xgb_model, X_test, feature_names, max_samples=2000
)

top_features_df = get_top_features(shap_values, feature_names, top_n=15)
print('\nTop 15 features by mean |SHAP value|:')
print(top_features_df.to_string(index=False))

In [ ]:
# SHAP summary bar plot
plt.figure(figsize=(9, 6))
shap.plots.bar(shap_explanation, max_display=15, show=False)
plt.title('XGBoost — Global Feature Importance (SHAP)')
plt.tight_layout()
plt.show()

In [ ]:
# SHAP beeswarm plot — shows direction of effect
plt.figure(figsize=(10, 7))
shap.plots.beeswarm(shap_explanation, max_display=15, show=False)
plt.title('XGBoost — SHAP Beeswarm (Feature Direction & Magnitude)')
plt.tight_layout()
plt.show()

> **SHAP Interpretation:**
> - `velocity_last_24h` — high values push the score up strongly (fraud signal).
> - `location_mismatch` — being set to 1 raises the fraud score significantly.
> - `previous_fraud_flag` — prior fraud history is a powerful predictor.
> - `log_amount` — large amounts increase fraud probability.
> - `risk_signal_count` — composite risk is among the most influential features.
> - `account_tenure_days` — low tenure increases predicted fraud risk.

> These findings align with domain knowledge and the patterns observed in EDA — reinforcing model credibility.

## 8. Confusion Matrix — Best Model (XGBoost)

In [ ]:
import numpy as np
from sklearn.metrics import ConfusionMatrixDisplay

xgb_preds = (xgb_probs >= xgb_metrics.threshold).astype(int)

fig, ax = plt.subplots(figsize=(6, 5))
disp = ConfusionMatrixDisplay(
    confusion_matrix=xgb_metrics.confusion,
    display_labels=['Legitimate', 'Fraud']
)
disp.plot(ax=ax, colorbar=False, cmap='Blues')
ax.set_title(f'XGBoost Confusion Matrix\n(threshold = {xgb_metrics.threshold:.2f})')
plt.tight_layout()
plt.show()

tn, fp, fn, tp = xgb_metrics.confusion.ravel()
print(f'True Positives  (fraud caught)    : {tp:,}')
print(f'False Positives (legitimate blocked): {fp:,}')
print(f'False Negatives (fraud missed)    : {fn:,}')
print(f'True Negatives  (legitimate passed): {tn:,}')

## 9. Model Selection

**Selected model: XGBoost**

| Criterion | Logistic Regression | Random Forest | XGBoost |
|-----------|--------------------|--------------|---------|
| AUC-ROC | Lowest | Middle | **Highest** |
| KS Statistic | Lowest | Middle | **Highest** |
| Recall | Competitive | Middle | **Best** |
| Score separation | Fair | Good | **Excellent** |
| Training speed | Fast | Moderate | Moderate |
| Interpretability | High | Medium | Medium (SHAP) |

XGBoost's superiority on recall and score separation makes it the best candidate for the fraud strategy in Notebook 04. The scores will be used directly as inputs to the strategy cutoff framework.

Logistic Regression serves as a useful **champion-challenger alternative** when regulatory interpretability requirements demand a simpler model.

In [ ]:
# Persist scores for strategy notebook
import joblib
from pathlib import Path

scores_df = pd.DataFrame({
    'transaction_id': test_df['transaction_id'].values,
    'fraud_score': xgb_probs,
    'is_fraud': y_test,
    'transaction_amount': test_df['transaction_amount'].values,
})

scores_df.to_csv('../data/test_scores.csv', index=False)
print(f'Test scores saved. Shape: {scores_df.shape}')
print(scores_df.head())